# 01 - Google Trends Ingestion

## Purpose
This notebook loads the expanded Google Trends raw dataset for the Coffee Dashboard project, validates it, and transforms it into Power BI-ready tables.

## Current status
The notebook currently covers:
- raw file loading and inspection
- fact table transformation (wide to long)
- fact table validation checks
- date dimension and trend dimension generation

---

### Import libraries, define project paths and load the expanded raw Google Trends dataset

In [23]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

RAW_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILE = RAW_DIR / "google_trends_expanded.csv"

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", RAW_DIR)
print("Interim data folder:", INTERIM_DIR)
print("Raw file exists:", RAW_FILE.exists())
print("Raw file path:", RAW_FILE)

df_raw = pd.read_csv(RAW_FILE)

df_raw.head()

Project root: C:\Users\lisan\OneDrive\Bureau\work\Projects\coffee-dashboard
Raw data folder: C:\Users\lisan\OneDrive\Bureau\work\Projects\coffee-dashboard\data\raw
Interim data folder: C:\Users\lisan\OneDrive\Bureau\work\Projects\coffee-dashboard\data\interim
Raw file exists: True
Raw file path: C:\Users\lisan\OneDrive\Bureau\work\Projects\coffee-dashboard\data\raw\google_trends_expanded.csv


,date,espresso,americano,cappuccino,latte,flat white,mocha,iced coffee,cold brew,iced latte,...,frappuccino,caramel frappuccino,mocha frappuccino,caramel macchiato,iced caramel macchiato,dalgona coffee,mushroom coffee,pour over coffee,french press coffee,drip coffee
0,2006-04-01,14,18,3,7,0,17,3,0,0,...,2,0,0,0,0,0,0,0,1,4
1,2006-05-01,14,19,3,7,0,15,3,1,0,...,2,0,0,0,0,0,0,0,1,5
2,2006-06-01,14,18,3,8,0,16,4,1,1,...,3,0,0,0,0,0,0,0,1,5
3,2006-07-01,13,17,3,7,0,17,6,1,1,...,3,0,0,0,0,0,0,0,1,4
4,2006-08-01,14,19,3,8,0,16,5,0,0,...,2,0,0,0,0,0,0,0,1,5


In [24]:
df_raw.info()
df_raw.columns.tolist()

<class 'pandas.DataFrame'>
RangeIndex: 241 entries, 0 to 240
Data columns (total 27 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   date                    241 non-null    str  
 1   espresso                241 non-null    int64
 2   americano               241 non-null    int64
 3   cappuccino              241 non-null    int64
 4   latte                   241 non-null    int64
 5   flat white              241 non-null    int64
 6   mocha                   241 non-null    int64
 7   iced coffee             241 non-null    int64
 8   cold brew               241 non-null    int64
 9   iced latte              241 non-null    int64
 10  iced americano          241 non-null    int64
 11  vanilla latte           241 non-null    int64
 12  caramel latte           241 non-null    int64
 13  hazelnut latte          241 non-null    int64
 14  pumpkin spice latte     241 non-null    int64
 15  peppermint mocha        241 non-nu

['date',
 'espresso',
 'americano',
 'cappuccino',
 'latte',
 'flat white',
 'mocha',
 'iced coffee',
 'cold brew',
 'iced latte',
 'iced americano',
 'vanilla latte',
 'caramel latte',
 'hazelnut latte',
 'pumpkin spice latte',
 'peppermint mocha',
 'gingerbread latte',
 'frappuccino',
 'caramel frappuccino',
 'mocha frappuccino',
 'caramel macchiato',
 'iced caramel macchiato',
 'dalgona coffee',
 'mushroom coffee',
 'pour over coffee',
 'french press coffee',
 'drip coffee']

---

### Transform the raw wide dataset into the fact table

In [25]:
df_fact = df_raw.melt(
    id_vars="date",
    var_name="keyword",
    value_name="search_interest"
)

df_fact["date"] = pd.to_datetime(df_fact["date"])
df_fact["search_interest"] = pd.to_numeric(df_fact["search_interest"])

df_fact = df_fact.sort_values(["keyword", "date"]).reset_index(drop=True)

df_fact.head

<bound method NDFrame.head of            date        keyword  search_interest
0    2006-04-01      americano               18
1    2006-05-01      americano               19
2    2006-06-01      americano               18
3    2006-07-01      americano               17
4    2006-08-01      americano               19
...         ...            ...              ...
6261 2025-12-01  vanilla latte               29
6262 2026-01-01  vanilla latte               34
6263 2026-02-01  vanilla latte               32
6264 2026-03-01  vanilla latte               33
6265 2026-04-01  vanilla latte               35

[6266 rows x 3 columns]>

In [26]:
df_fact.info()

<class 'pandas.DataFrame'>
RangeIndex: 6266 entries, 0 to 6265
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   date             6266 non-null   datetime64[us]
 1   keyword          6266 non-null   str           
 2   search_interest  6266 non-null   int64         
dtypes: datetime64[us](1), int64(1), str(1)
memory usage: 147.0 KB


In [27]:
OUTPUT_FILE = INTERIM_DIR / "google_trends_fact.csv"
df_fact.to_csv(OUTPUT_FILE, index=False)

print(f"Saved fact table to: {OUTPUT_FILE}")

Saved fact table to: C:\Users\lisan\OneDrive\Bureau\work\Projects\coffee-dashboard\data\interim\google_trends_fact.csv


In [28]:
# Run checks on the fact table

# Basic structure
print("Fact table shape:", df_fact.shape)
print("\nColumns:")
print(df_fact.columns.tolist())

# Null checks
print("\nNull values by column:")
print(df_fact.isna().sum())

# Duplicate checks
duplicate_count = df_fact.duplicated().sum()
print("\nNumber of duplicated rows:", duplicate_count)

# Check keyword list
print("\nKeywords:")
print(sorted(df_fact["keyword"].dropna().unique()))

# Check date range
print("\nMin date:", df_fact["date"].min())
print("Max date:", df_fact["date"].max())

# Search interest range
print("\nMinimum search interest:", df_fact["search_interest"].min())
print("Maximum search interest:", df_fact["search_interest"].max())

Fact table shape: (6266, 3)

Columns:
['date', 'keyword', 'search_interest']

Null values by column:
date               0
keyword            0
search_interest    0
dtype: int64

Number of duplicated rows: 0

Keywords:
['americano', 'cappuccino', 'caramel frappuccino', 'caramel latte', 'caramel macchiato', 'cold brew', 'dalgona coffee', 'drip coffee', 'espresso', 'flat white', 'frappuccino', 'french press coffee', 'gingerbread latte', 'hazelnut latte', 'iced americano', 'iced caramel macchiato', 'iced coffee', 'iced latte', 'latte', 'mocha', 'mocha frappuccino', 'mushroom coffee', 'peppermint mocha', 'pour over coffee', 'pumpkin spice latte', 'vanilla latte']

Min date: 2006-04-01 00:00:00
Max date: 2026-04-01 00:00:00

Minimum search interest: 0
Maximum search interest: 100


---

### Create and export the date dimension

In [29]:
min_date = df_fact["date"].min()
max_date = df_fact["date"].max()

date_range = pd.date_range(start=min_date, end=max_date, freq="D")

dim_date = pd.DataFrame({"date": date_range})

dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter"] = "Q" + dim_date["date"].dt.quarter.astype(str)
dim_date["month_number"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.strftime("%B")
dim_date["year_month"] = dim_date["date"].dt.strftime("%Y-%m")
dim_date["month_year_label"] = dim_date["date"].dt.strftime("%B %Y")
dim_date["season"] = dim_date["month_number"].map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn"})
dim_date["week_of_year"] = dim_date["date"].dt.isocalendar().week.astype(int)
dim_date["day_of_month"] = dim_date["date"].dt.day
dim_date["day_name"] = dim_date["date"].dt.strftime("%A")

dim_date.head()

,date,year,quarter,month_number,month_name,year_month,month_year_label,season,week_of_year,day_of_month,day_name
0,2006-04-01,2006,Q2,4,April,2006-04,April 2006,Spring,13,1,Saturday
1,2006-04-02,2006,Q2,4,April,2006-04,April 2006,Spring,13,2,Sunday
2,2006-04-03,2006,Q2,4,April,2006-04,April 2006,Spring,14,3,Monday
3,2006-04-04,2006,Q2,4,April,2006-04,April 2006,Spring,14,4,Tuesday
4,2006-04-05,2006,Q2,4,April,2006-04,April 2006,Spring,14,5,Wednesday


In [30]:
dim_date.info()

<class 'pandas.DataFrame'>
RangeIndex: 7306 entries, 0 to 7305
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              7306 non-null   datetime64[us]
 1   year              7306 non-null   int32         
 2   quarter           7306 non-null   str           
 3   month_number      7306 non-null   int32         
 4   month_name        7306 non-null   str           
 5   year_month        7306 non-null   str           
 6   month_year_label  7306 non-null   str           
 7   season            7306 non-null   str           
 8   week_of_year      7306 non-null   int64         
 9   day_of_month      7306 non-null   int32         
 10  day_name          7306 non-null   str           
dtypes: datetime64[us](1), int32(3), int64(1), str(6)
memory usage: 542.4 KB


In [31]:
DATE_DIM_FILE = INTERIM_DIR / "dim_date.csv"
dim_date.to_csv(DATE_DIM_FILE, index=False)

print(f"Saved date dimension to: {DATE_DIM_FILE}")
print("Shape:", dim_date.shape)

Saved date dimension to: C:\Users\lisan\OneDrive\Bureau\work\Projects\coffee-dashboard\data\interim\dim_date.csv
Shape: (7306, 11)


---

### Create and export the trend dimension

In [32]:
dim_trend = pd.DataFrame({
    "keyword": df_fact["keyword"].dropna().drop_duplicates().sort_values().reset_index(drop=True)
})

# Mapping
trend_metadata = {
    # Core hot coffee drinks
    "espresso": {"category": "coffee", "style": "espresso", "serving_style": "hot", "trend_type": "core", "contains_milk": False},
    "americano": {"category": "coffee", "style": "americano", "serving_style": "hot", "trend_type": "core", "contains_milk": False},
    "cappuccino": {"category": "coffee", "style": "cappuccino", "serving_style": "hot", "trend_type": "core", "contains_milk": True},
    "latte": {"category": "coffee", "style": "latte", "serving_style": "hot", "trend_type": "core", "contains_milk": True},
    "flat white": {"category": "coffee", "style": "flat_white", "serving_style": "hot", "trend_type": "core", "contains_milk": True},
    "mocha": {"category": "coffee", "style": "mocha", "serving_style": "hot", "trend_type": "core", "contains_milk": True},

    # Core cold coffee drinks
    "iced coffee": {"category": "coffee", "style": "iced_coffee", "serving_style": "cold", "trend_type": "core", "contains_milk": False},
    "cold brew": {"category": "coffee", "style": "cold_brew", "serving_style": "cold", "trend_type": "core", "contains_milk": False},
    "iced latte": {"category": "coffee", "style": "latte", "serving_style": "cold", "trend_type": "core", "contains_milk": True},
    "iced americano": {"category": "coffee", "style": "americano", "serving_style": "cold", "trend_type": "core", "contains_milk": False},

    # Flavoured lattes
    "vanilla latte": {"category": "coffee", "style": "latte", "serving_style": "hot", "trend_type": "core", "contains_milk": True},
    "caramel latte": {"category": "coffee", "style": "latte", "serving_style": "hot", "trend_type": "core", "contains_milk": True},
    "hazelnut latte": {"category": "coffee", "style": "latte", "serving_style": "hot", "trend_type": "core", "contains_milk": True},

    # Seasonal or limited-time offerings
    "pumpkin spice latte": {"category": "coffee", "style": "latte", "serving_style": "hot", "trend_type": "seasonal", "contains_milk": True},
    "peppermint mocha": {"category": "coffee", "style": "mocha", "serving_style": "hot", "trend_type": "seasonal", "contains_milk": True},
    "gingerbread latte": {"category": "coffee", "style": "latte", "serving_style": "hot", "trend_type": "seasonal", "contains_milk": True},

    # Frappuccinos / desserts
    "frappuccino": {"category": "coffee", "style": "frappuccino", "serving_style": "cold", "trend_type": "core", "contains_milk": True},
    "caramel frappuccino": {"category": "coffee", "style": "frappuccino", "serving_style": "cold", "trend_type": "core", "contains_milk": True},
    "mocha frappuccino": {"category": "coffee", "style": "frappuccino", "serving_style": "cold", "trend_type": "core", "contains_milk": True},

    # Macchiato drinks
    "caramel macchiato": {"category": "coffee", "style": "latte", "serving_style": "hot", "trend_type": "core", "contains_milk": True},
    "iced caramel macchiato": {"category": "coffee", "style": "latte", "serving_style": "cold", "trend_type": "core", "contains_milk": True},

    # Niche or emerging trends
    "dalgona coffee": {"category": "coffee", "style": "whipped_coffee", "serving_style": "cold", "trend_type": "trendy", "contains_milk": False},
    "mushroom coffee": {"category": "coffee", "style": "functional_coffee", "serving_style": "hot", "trend_type": "niche", "contains_milk": False},

    # Other brewing methods
    "pour over coffee": {"category": "coffee", "style": "pour_over", "serving_style": "hot", "trend_type": "niche", "contains_milk": False},
    "french press coffee": {"category": "coffee", "style": "french_press", "serving_style": "hot", "trend_type": "niche", "contains_milk": False},
    "drip coffee": {"category": "coffee", "style": "drip_coffee", "serving_style": "hot", "trend_type": "core", "contains_milk": False},
}

dim_trend["category"] = dim_trend["keyword"].map(lambda x: trend_metadata.get(x, {}).get("category"))
dim_trend["style"] = dim_trend["keyword"].map(lambda x: trend_metadata.get(x, {}).get("style"))
dim_trend["serving_style"] = dim_trend["keyword"].map(lambda x: trend_metadata.get(x, {}).get("serving_style"))
dim_trend["trend_type"] = dim_trend["keyword"].map(lambda x: trend_metadata.get(x, {}).get("trend_type"))
dim_trend["contains_milk"] = dim_trend["keyword"].map(lambda x: trend_metadata.get(x, {}).get("contains_milk"))

dim_trend.head()

,keyword,category,style,serving_style,trend_type,contains_milk
0,americano,coffee,americano,hot,core,False
1,cappuccino,coffee,cappuccino,hot,core,True
2,caramel frappuccino,coffee,frappuccino,cold,core,True
3,caramel latte,coffee,latte,hot,core,True
4,caramel macchiato,coffee,latte,hot,core,True


In [33]:
TREND_DIM_FILE = INTERIM_DIR / "dim_trend.csv"
dim_trend.to_csv(TREND_DIM_FILE, index=False)

print(f"Saved trend dimension to: {TREND_DIM_FILE}")

Saved trend dimension to: C:\Users\lisan\OneDrive\Bureau\work\Projects\coffee-dashboard\data\interim\dim_trend.csv
